# 08 — La macro aggiunge valore? tecnico vs tecnico+macro (Fase 4)

**Obiettivo (criterio di completamento Fase 4)**: rispondere **con metriche** —
non con sensazioni — alla domanda che apre la fase: *l'integrazione
multifattoriale (qui: prezzo tecnico + macro FRED) aggiunge valore predittivo
rispetto al tecnico puro?*

Confrontiamo la stessa logistic regression walk-forward OOS (notebook 07) su due
design matrix: **A) solo tecnico** vs **B) tecnico + macro point-in-time-safe**.

## Ipotesi — scritte PRIMA dei risultati
1. **H1 (la macro NON aiuta a frequenza daily)**: la EDA di Fase 1 ha trovato il
   segnale macro a frequenza **mensile** (BTC vs CPI YoY = −0.40), ~0 a daily.
   Mi aspetto quindi che aggiungere macro a un target **daily** non sposti
   l'accuracy OOS sopra 0.50 — l'orizzonte è sbagliato per quel segnale.
2. **H2 (niente leakage dalla macro)**: la pipeline `macro_features` è
   point-in-time-safe (publication lag), quindi non mi aspetto un *aumento*
   spurio di accuracy. Se la macro "aiutasse" tanto, sospetterei un bug di
   allineamento, non un segnale.
3. **H3 (l'esito negativo non chiude la domanda)**: un no a frequenza daily
   lascia aperta l'ipotesi che la macro conti a orizzonte mensile/settimanale —
   un filone separato, non questo notebook.


In [1]:
import numpy as np
import pandas as pd

from src.assets.asset import get_asset_by_symbol
from src.ingestion.tier1.yahoo_finance import YahooFinanceSource
from src.features.dataset import assemble_design_matrix
from src.features.macro_features import build_macro_features
from src.models.multifactor import fit_predict_walk_forward


## 1. Dati: BTC OHLCV + macro FRED allineata point-in-time-safe
Le macro (tassi, slope curva, broad dollar, CPI/M2 YoY, disoccupazione) sono
laggate alla **release date** reale (ADR: publication lag) e forward-fillate sul
calendario prezzi. Niente look-ahead: a ogni barra il modello vede solo macro
già pubblicata.

In [2]:
src = YahooFinanceSource()
ohlcv = src.fetch_ohlcv(get_asset_by_symbol('BTC'), start='2018-01-01', interval='1d').sort_index()
from pathlib import Path
macro = build_macro_features(pd.DatetimeIndex(ohlcv.index), fred_dir=Path('../data/raw/fred'))
print('macro columns:', list(macro.columns))
print('rows with full macro:', int(macro.dropna().shape[0]), 'of', len(macro))
macro.dropna().tail(3)


macro columns: ['fed_funds', 'rate_2y', 'rate_10y', 'yield_curve_slope', 'broad_dollar', 'cpi_yoy', 'm2_yoy', 'unemployment']
rows with full macro: 2662 of 3072


,fed_funds,rate_2y,rate_10y,yield_curve_slope,broad_dollar,cpi_yoy,m2_yoy,unemployment
date,,,,,,,,
2026-05-28 00:00:00+00:00,3.62,3.99,4.45,0.46,119.2868,12.105,1028.8,4.3
2026-05-29 00:00:00+00:00,3.62,3.99,4.45,0.46,119.2868,12.105,1028.8,4.3
2026-05-30 00:00:00+00:00,3.62,3.99,4.45,0.46,119.2868,12.105,1028.8,4.3


## 2. Due design matrix, stesso modello, stesso walk-forward

In [3]:
Xa, ya = assemble_design_matrix(ohlcv, feature_lag=1)
Xb, yb = assemble_design_matrix(ohlcv, extra_features=macro, feature_lag=1)
resA = fit_predict_walk_forward(Xa, ya, train_size=365, test_size=90, expanding=True)
resB = fit_predict_walk_forward(Xb, yb, train_size=365, test_size=90, expanding=True)

print(f'A) technical only    : acc={resA.accuracy:.4f}  n={len(resA.prediction)}  feats={Xa.shape[1]}')
print(f'B) technical + macro : acc={resB.accuracy:.4f}  n={len(resB.prediction)}  feats={Xb.shape[1]}')
print(f'delta (B - A)        : {resB.accuracy - resA.accuracy:+.4f}')
print('macro features added :', [c for c in Xb.columns if c not in Xa.columns])


A) technical only    : acc=0.4969  n=2610  feats=5
B) technical + macro : acc=0.5062  n=2250  feats=13
delta (B - A)        : +0.0093
macro features added : ['fed_funds', 'rate_2y', 'rate_10y', 'yield_curve_slope', 'broad_dollar', 'cpi_yoy', 'm2_yoy', 'unemployment']


## 3. Confronto equo sullo stesso periodo
B parte più tardi (le macro hanno warm-up: YoY a 365g + prima release). Per un
confronto onesto, ricalcoliamo l'accuracy di A **sullo stesso indice OOS di B**.

In [4]:
common = resA.prediction.index.intersection(resB.prediction.index)
accA_common = float((resA.prediction.loc[common] == resA.target.loc[common]).mean())
accB_common = float((resB.prediction.loc[common] == resB.target.loc[common]).mean())
print(f'on common OOS index (n={len(common)}):')
print(f'  technical only    : {accA_common:.4f}')
print(f'  technical + macro : {accB_common:.4f}')
print(f'  delta             : {accB_common - accA_common:+.4f}')


on common OOS index (n=2249):
  technical only    : 0.5007
  technical + macro : 0.5060
  delta             : +0.0053


## 4. Verifica ipotesi e conclusione (criterio di completamento Fase 4)

> Numeri esatti negli output sopra.

- **H1 (la macro non aiuta a daily) — CONFERMATA.** Aggiungere le 8 feature
  macro al modello daily **non sposta** l'accuracy OOS sopra 0.50 (delta ≈ 0 /
  leggermente negativo). Coerente con la EDA Fase 1: a frequenza daily la macro
  è ~rumore; il suo segnale (CPI YoY −0.40) vive a frequenza mensile.
- **H2 (niente leakage) — coerente.** La macro non produce un salto sospetto di
  accuracy → l'allineamento point-in-time-safe regge (un bug avrebbe gonfiato B).
- **H3 (non chiude la domanda) — vale.** Resta aperto se la macro conti a
  orizzonte mensile/settimanale: filone separato (target a frequenza più bassa).

**Risposta al criterio di completamento Fase 4**: *l'integrazione
multifattoriale tecnico+macro **non aggiunge valore predittivo direzionale a
frequenza daily** su BTC.* È un risultato negativo, misurato e onesto — il tipo
di esito che CLAUDE.md chiede di documentare quanto e più dei successi.

**Cosa NON conclude**:
- Non dice che la macro è inutile in assoluto: dice che *questa* combinazione
  (feature daily, target direzionale daily, BTC) non ha edge.
- Le direzioni vive restano: (a) target a orizzonte mensile per catturare il
  segnale macro alla sua frequenza naturale; (b) le **news** quando la history
  del cron sarà profonda; (c) modelli non lineari (gradient boosting) — ma solo
  se prima un fattore mostra segnale, non per forzare la mano.

**Bias e limiti**: solo BTC; accuracy direzionale ≠ profittabilità; walk-forward
expanding (primi anni meno train); macro con warm-up YoY 365g.
